# Tutorial 6: Amplitude Estimation Variants

This notebook compares the four QAE variants in qufin:
1. **Canonical QAE** -- QPE-based, requires controlled-Grover and QFT
2. **IQAE** -- Iterative, no QFT, adaptive schedule
3. **MLAE** -- Maximum likelihood, fixed-schedule measurements
4. **FQAE** -- Fourier-based, low circuit depth

**References**: Brassard et al. (2002), Grinko et al. (2021), Suzuki et al. (2020), Giurgica-Tiron et al. (2022).

In [ ]:
import numpy as np
import time
np.random.seed(42)

## 1. Common Problem Setup

We price a European call option with all four methods.

In [ ]:
from qufin.options.amplitude_estimation.european_qae import (
    EuropeanQAESpec, build_european_estimation_problem,
)
from qufin.options.classical.black_scholes import bs_price
from qufin.backends.qiskit_backend import QiskitAerBackend

S, K, sigma, r, T = 100, 105, 0.2, 0.05, 1.0
bs_ref = bs_price(s=S, k=K, sigma=sigma, r=r, T=T, option_type="call")

spec = EuropeanQAESpec(
    s=S, k=K, sigma=sigma, r=r, T=T,
    n_qubits=5, option_type="call",
)
problem = build_european_estimation_problem(spec)
backend = QiskitAerBackend(shots=4096)

print(f"Black-Scholes reference: ${bs_ref:.4f}")

## 2. Canonical QAE (QPE-Based)

In [ ]:
from qufin.options.amplitude_estimation.canonical import (
    CanonicalQAE,
)

t0 = time.time()
canonical = CanonicalQAE(
    problem=problem,
    backend=backend,
    n_eval_qubits=4,
)
canonical_result = canonical.estimate()
t_canonical = time.time() - t0

print(f"Canonical QAE: ${canonical_result.value:.4f}  (error={abs(canonical_result.value - bs_ref):.4f}, time={t_canonical:.2f}s)")

## 3. Iterative QAE (IQAE)

In [ ]:
from qufin.options.amplitude_estimation.iqae import (
    IQAEConfig, IterativeAmplitudeEstimation,
)

t0 = time.time()
iqae = IterativeAmplitudeEstimation(
    problem=problem,
    backend=backend,
    config=IQAEConfig(epsilon_target=0.01, alpha=0.05),
)
iqae_result = iqae.estimate()
t_iqae = time.time() - t0

print(f"IQAE:          ${iqae_result.value:.4f}  (error={abs(iqae_result.value - bs_ref):.4f}, time={t_iqae:.2f}s)")

## 4. Maximum Likelihood QAE (MLAE)

In [ ]:
from qufin.options.amplitude_estimation.mlae import (
    MLAEConfig, MaximumLikelihoodAE,
)

t0 = time.time()
mlae = MaximumLikelihoodAE(
    problem=problem,
    backend=backend,
    config=MLAEConfig(max_power=4),
)
mlae_result = mlae.estimate()
t_mlae = time.time() - t0

print(f"MLAE:          ${mlae_result.value:.4f}  (error={abs(mlae_result.value - bs_ref):.4f}, time={t_mlae:.2f}s)")

## 5. Fourier QAE (FQAE)

In [ ]:
from qufin.options.amplitude_estimation.fqae import (
    FQAEConfig, FourierQAE,
)

t0 = time.time()
fqae = FourierQAE(
    problem=problem,
    backend=backend,
    config=FQAEConfig(max_power=4),
)
fqae_result = fqae.estimate()
t_fqae = time.time() - t0

print(f"FQAE:          ${fqae_result.value:.4f}  (error={abs(fqae_result.value - bs_ref):.4f}, time={t_fqae:.2f}s)")

## 6. Comparison Table

In [ ]:
results = [
    ("Canonical", canonical_result.value, t_canonical),
    ("IQAE", iqae_result.value, t_iqae),
    ("MLAE", mlae_result.value, t_mlae),
    ("FQAE", fqae_result.value, t_fqae),
]

print(f"{'Method':<12} {'Price':>8} {'Error':>8} {'Time (s)':>10}")
print("-" * 42)
print(f"{'BS (ref)':<12} {bs_ref:>8.4f} {0.0:>8.4f} {'--':>10}")
for name, price, t in results:
    print(f"{name:<12} {price:>8.4f} {abs(price - bs_ref):>8.4f} {t:>10.2f}")

## Summary

| Method | QPE Required | Circuit Depth | Key Property |
|:-------|:-------------|:-------------|:-------------|
| Canonical | Yes | Deep (QFT + c-Grover) | Optimal query complexity |
| IQAE | No | Moderate (adaptive) | Near-optimal, no QFT |
| MLAE | No | Moderate (fixed schedule) | Robust to noise |
| FQAE | No | Shallow | Best for near-term hardware |

**Next**: Tutorial 07 applies QAE to risk management (VaR/CVaR).